# NIH metrics

This notebook uses information extracted from [NIH Exporter](https://reporter.nih.gov/exporter) to identify NIH-funded users of PhysioNet.

## Setup

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from rapidfuzz.distance import JaroWinkler

from twentyfiveyears.nih import (combine_exporter_tables, get_pi_names, get_authors, link_users)

In [2]:
# Set the base path
base_path = os.path.join("..", "data")

## Load map of Person IDs

All users are assigned a unique `person_id`.

In [3]:
# Load the map of Person IDs
path = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
person_map = pd.read_csv(path)
person_map.head(3)

,person_id,physionet_id
0,100000000,2
1,100000001,6
2,100000002,8


## Load PhysioNet dataset

Load a dataset containing the list of PhysioNet users

In [4]:
# Load DataFrame of PhysioNet users
path = os.path.join(base_path, 'physionet', 'users.csv')
physionet_users = pd.read_csv(path, low_memory=False)

In [5]:
physionet_users.head(3)

,user_id,username,join_date,last_login,registration_ip,is_active_user,primary_email,all_emails,first_names,last_name,...,credentialing_job_title,credentialing_city,credentialing_state_or_province,credentialing_country,credentialing_webpage,credentialing_reference_name,credentialing_reference_email,credentialing_reference_org,credentialing_reference_response,credentialing_research_summary
0,2,ftorres,2019-01-11,2024-05-30 23:51:55.929525+00:00,NaN,True,ftf.dummy@gmail.com,"ftorres@physionet.org, ftf.dummy@gmail.com",Felipe III,Torres Fábregas,...,TEST,TEST,TEST,US,NaN,TEST,felipe.torres.cs@gmail.com,NaN,NaN,TEST
1,6,tompollard,2019-02-19,2024-08-01 15:22:27.844114+00:00,NaN,True,tpollard@mit.edu,"tpollard@mit.edu, tompollardx@gmail.com",Tom,Pollard,...,Research Scientist,Somerville,MA,US,NaN,Tom Pollard,tpollard@mit.edu,NaN,Me.,Laboratory for Computational Physiology
2,8,benjamin,2019-02-21,2024-08-06 20:16:01.888720+00:00,NaN,True,bmoody@mit.edu,"bmoody@mit.edu, benjaminmoody+test123@gmail.co...",Benjamin,Moody,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Now join to the list of Person IDs, and drop unused columns:

In [6]:
# Add person_id column to the users table
physionet_users = pd.merge(physionet_users, person_map, left_on='user_id', right_on='physionet_id')

# Only keep the columns we need from the users table
physionet_users = physionet_users[['person_id', 'full_name']].copy()

# Rename the 'full_name' column to 'physionet_name'
physionet_users = physionet_users.rename(columns={'full_name': 'physionet_name'})

physionet_users.head(3)

,person_id,physionet_name
0,100000000,Felipe III Torres Fábregas
1,100000001,Tom Pollard
2,100000002,Benjamin Moody


## Load Principal Investigators of NIH projects

Load a list of Principal Investigators

In [7]:
# Load the NIH project data
path = os.path.join(base_path, 'nih', 'exporter', 'projects')
projects = combine_exporter_tables(path, "RePORTER_PRJ_C_FY", start_year=1995)
projects.head(3)

,APPLICATION_ID,ACTIVITY,ADMINISTERING_IC,APPLICATION_TYPE,ARRA_FUNDED,AWARD_NOTICE_DATE,BUDGET_START,BUDGET_END,CFDA_CODE,CORE_PROJECT_NUM,...,SUBPROJECT_ID,SUFFIX,SUPPORT_YEAR,TOTAL_COST,TOTAL_COST_SUB_PROJECT,OPPORTUNITY NUMBER,FUNDING_MECHANISM,ORG_IPF_CODE,DIRECT_COST_AMT,INDIRECT_COST_AMT
0,2056372,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001117,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2056373,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001118,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2056374,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001119,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Get the names of Principal Investigators
investigators = get_pi_names(projects)
investigators[0:3]

['eli i capilouto', 'abdelmonem a afifi', 'richard h hart']

## Load authors of publications linked to NIH projects

Load a list of authors linked to NIH projects

In [9]:
# Load the NIH publications data
path = os.path.join(base_path, 'nih', 'exporter', 'publications')
publications = combine_exporter_tables(path, "RePORTER_PUB_C_", start_year=1995)
publications.head(3)

,AFFILIATION,AUTHOR_LIST,COUNTRY,ISSN,JOURNAL_ISSUE,JOURNAL_TITLE,JOURNAL_TITLE_ABBR,JOURNAL_VOLUME,LANG,PAGE_NUMBER,PMC_ID,PMID,PUB_DATE,PUB_TITLE,PUB_YEAR
0,"Department of Fisheries and Wildlife, Oregon S...","Curtis, L R; Zhang, Q; el-Zahr, C; Carpenter, ...",UNITED STATES,0272-0590,1,Fundamental and applied toxicology : official...,Fundam Appl Toxicol,25,eng,146-53,NaN,7601322,1995 Apr,Temperature-modulated incidence of aflatoxin B...,1995
1,"Department of Biology, Boston University, Mass...","Loechler, E L",UNITED STATES,0899-1987,4,Molecular carcinogenesis.,Mol Carcinog,13,eng,213-9,NaN,7646760,1995 Aug,How are potent bulky carcinogens able to induc...,1995
2,Department of Pharmacology and Toxicology Scho...,"Carlson, G P; Olson, R M",AUSTRALIA,1039-9712,1,Biochemistry and molecular biology internation...,Biochem Mol Biol Int,37,eng,65-71,NaN,8653089,1995 Sep,Comparison of the metabolism of alcohols by ra...,1995


In [10]:
# Get the names of authors
authors = get_authors(publications)
authors[0:3]

['l r curtis', 'q zhang', 'c el-zahr']

## Match NIH listed people to PhysioNet users

Attempt to match people between the two sources

In [14]:
# Match NIH Principal Investigators to PhysioNet users
# Set limit for testing
limit = 100
physionet_users = link_users(physionet_users, investigators, match_group="investigators", limit=limit)

100%|█████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 5957.65it/s]

Finished in 0.03363323211669922 seconds


In [15]:
# Match NIH authors to PhysioNet users
physionet_users = link_users(physionet_users, authors, match_group="authors", limit=limit)

100%|█████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 4928.27it/s]

Finished in 0.033467769622802734 seconds


In [16]:
physionet_users.head(5)

,person_id,physionet_name,matched_investigator_score,matched_investigator_name,matched_author_score,matched_author_name
0,100000000,Felipe III Torres Fábregas,0.653846,delois p. weekes,0.619231,l r curtis
1,100000001,Tom Pollard,0.636364,rose porter,0.626263,m a apicella
2,100000002,Benjamin Moody,0.616356,marilyn frank-stromborg,0.620635,p j cannon
3,100000003,Alistair Johnson,0.683333,patricia dolphin,0.655429,r johannsdottir
4,100000004,Julian Euma Ishii-Rousseau,0.689005,james h ware,0.685897,j m lash


Save the results

## Save the results

In [39]:
# Save the results
save_path = os.path.join(base_path, 'physionet_users_nih_funded.csv')
path = Path(save_path)

# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()

# Output the merged DataFrame or save it to a file
physionet_users.to_csv(normalized_path, index=False)